# Bibliotecas

## Sistema (os, makedirs, glob)

In [1]:
import os
from os import makedirs
from os import listdir
import glob

## Básicos (numpy, math, display, locale, time, random, re)

In [2]:
# !python -m pip install jupyter


# !python -m pip install IPython
from IPython.display import display

import math

# !python -m pip install numpy
import numpy as np

import locale
# locale.setlocale(locale.LC_ALL, "pt_BR.UTF-8")  # Use "" for auto, or force e.g. to "en_US.UTF-8"

import time
from datetime import datetime, timedelta, date

from pandas.tseries.offsets import BDay # para os dias úteis
# today = datetime.datetime.today()
# print(today - BDay(4)) # 4 dias úteis atrás

# # !python -m pip install random
import random
random.seed(42)

# !python -m pip install regex
import re

## Leitura e análise de dados (Excel, Pandas, Spark)

In [3]:
# !python -m pip install findspark

# !python -m pip install openpyxl
# import openpyxl

# !python -m pip install xlsxwriter
# import xlsxwriter

# !python -m pip install xlrd
# import xlrd

# !python -m pip install python-calamine
# import python_calamine


# !python -m pip install pandas
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
# pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Finanças (yfinance, mplfinance)

In [4]:
# https://pypi.org/project/yfinance/
# https://github.com/ranaroussi/yfinance/wiki/Ticker

!python -m pip install yfinance
import yfinance as yf

!python -m pip install mplfinance
import mplfinance as mpf

# # Em R
# # https://cran.r-project.org/web/packages/BatchGetSymbols/index.html

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


# Funções

### Agrupar cada coluna (_agrupamento_cada_coluna_)

In [5]:
def agrupamento_cada_coluna(
    bd, 
    coluna_completa, 
    colunas_ignoradas = [], 
    ascending = False,
    imprime_tabelas = True,
    ):
    
    from IPython.display import display

    campos_com_erro = []
    dict_campos = {}

    colunas = bd.columns.drop(coluna_completa)

    if len(colunas_ignoradas) > 0:
        colunas = colunas.drop(colunas_ignoradas)
    
    for coluna in colunas:
        try: 
            temp_coluna = bd.fillna("(vazio)").groupby(coluna).count()[[coluna_completa]].rename(columns = {coluna_completa: "Quantidade"}).sort_values("Quantidade", ascending = ascending)

            if ascending == False:
                temp_coluna["%"] = temp_coluna["Quantidade"]/temp_coluna["Quantidade"].sum()
                temp_coluna["% acumulado"] = temp_coluna["%"].cumsum()
            
            if imprime_tabelas == True:
                display(temp_coluna)
                
            dict_campos[coluna] = temp_coluna
            # limpa(temp_coluna)
        
        except:
            campos_com_erro.append(coluna)
            #print("Campo " + coluna + " deu erro =/")
    
    return [campos_com_erro, dict_campos]

### Elimina colunas NA (_elimina_colunas_NA_)

In [6]:
def elimina_colunas_NA(
    base,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False
    ):

    tamanho_da_base = len(base)

    colunas_vazias = []
    colunas_completas = []
    colunas_parciais = []

    for coluna in base.columns:
        tamanho_da_coluna = len(base[base[coluna].isna()])
        if tamanho_da_coluna == tamanho_da_base:
            # print(coluna)
            colunas_vazias.append(coluna)
        elif tamanho_da_coluna == 0:
            colunas_completas.append(coluna)
        else:
            colunas_parciais.append(coluna)

    if imprime_colunas_vazias == True:
        if len(colunas_vazias) == 0:
            print("Não há colunas NA")
        else:
            print("Colunas vazias: " + str(colunas_vazias))
            
    if imprime_colunas_completas == True:
        if len(colunas_completas) == 0:
            print("Não há colunas completas")
        else:
            print("Colunas completas: " + str(colunas_completas))
    
    if imprime_colunas_parciais == True:
        if len(colunas_parciais) == 0:
            print("Não há colunas parciais")
        else:
            print("Colunas parciais: " + str(colunas_parciais))

    if remove_da_base == True:
        base = base.drop(colunas_vazias, axis = 1)
    
    if retorna_parciais == True:
        return [base, colunas_parciais]
    else:
        return base

### Análise exploratória básica de todos os campos da base - MUITO ÚTIL (_analise_exploratoria_)

In [7]:
def analise_exploratoria(
    bd,
    imprime_todas_colunas = False,
    imprime_info_colunas = True,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False,
    detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    ):

    # COMEÇANDO PELAS COLUNAS DISPONÍVEIS E INFO
    if imprime_todas_colunas == True:
        display(bd.columns)
    
    if imprime_info_colunas == True:
        for i in range(int(np.ceil(len(bd.columns)/20))):
            display(bd.iloc[:, (i*20):min((i+1)*20, len(bd.columns))].info())

    # DETALHAMENTO DE QUAIS COLUNAS SÃO NA OU PARCIAIS
    if retorna_parciais == True:
        [bd_semNA, colunas_parciais] = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = retorna_parciais
        )

        if detalhar_colunas_parciais == True:
            for coluna in colunas_parciais:
                print("# " + coluna + ": " + str(len(colunas_parciais[colunas_parciais[coluna].isna()])))
    else:
        colunas_parciais = []

        bd_semNA = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = retorna_parciais
        )

    [campos_com_erro, colunas_agrupadas] = agrupamento_cada_coluna(
        bd.reset_index(), 
        coluna_completa = bd_semNA.drop(colunas_parciais, axis = 1).reset_index().columns[0],
        colunas_ignoradas = colunas_ignoradas
    )
    
    if retorna_parciais == True:
        return [bd_semNA, colunas_parciais, colunas_agrupadas, campos_com_erro]
    else:
        return [bd_semNA, colunas_agrupadas, campos_com_erro]

# Leitura dos dados

In [8]:
var_caminho = r"C:\Users\ricardopeloi\OneDrive - falconi365\Data Science\O_Mais_Novo_Day_Trader_do_Brasil\o_mais_novo_day_trader_do_brasil\Bases"
# var_arquivo = r"\Bases\Lista de ações Análise 2025-03-16.xlsx"

lista_arquivos = listdir(var_caminho)
lista_arquivos_analises = []
for arquivo in lista_arquivos:
    if arquivo.find(" Análise") > 0:
        lista_arquivos_analises.append(datetime.strptime(arquivo.split(" Análise ")[1].split(".xls")[0], "%Y-%m-%d"))

var_arquivo_mais_recente = "/Lista de ações Análise " + max(lista_arquivos_analises).strftime("%Y-%m-%d") + ".xlsx"
# print(var_arquivo_mais_recente)

bd_dados_completos = pd.read_excel(var_caminho + var_arquivo_mais_recente).set_index("Ticker")
bd_dados_completos

,Nome da Empresa,Volume no último dia útil (lido em 16/03/2025),industry,industryKey,industryDisp,sector,sectorKey,sectorDisp,fullTimeEmployees,dividendRate,...,2025-03-26; Close,2025-03-26; Volume,2025-03-26; Dividends,2025-03-26; Stock Splits,2025-03-26; HLC,2025-03-26; Ticker,Alfa HLC; últimos 13 dias,Alfa HLC; últimos 55 dias,Martelos,Tipos de Martelos
Ticker,,,,,,,,,,,,,,,,,,,,,
B3SA3,B3,126021900,Financial Data & Stock Exchanges,financial-data-stock-exchanges,Financial Data & Stock Exchanges,Financial Services,financial-services,Financial Services,NaN,0.30,...,12.38,22021200,0.06,0,12.44,B3SA3,0.18,0.03,"2025-02-03, 2025-02-10, 2025-02-11, 2025-02-12...","Descida, Subida, Descida, Subida, Descida, Sub..."
HAPV3,Hapvida,111755000,Insurance - Life,insurance-life,Insurance - Life,Financial Services,financial-services,Financial Services,NaN,NaN,...,2.23,69226000,0.00,0,2.24,HAPV3,0.01,-0.01,"2025-02-10, 2025-02-13, 2025-02-17, 2025-02-19...","Subida, Subida, Subida, Descida, Subida, Subid..."
NTCO3,Natura,105384100,Household & Personal Products,household-personal-products,Household & Personal Products,Consumer Defensive,consumer-defensive,Consumer Defensive,NaN,0.74,...,10.06,25471700,0.00,0,10.09,NTCO3,-0.34,-0.10,"2025-02-20, 2025-03-07, 2025-03-11, 2025-03-13...","Subida, Subida, Descida, Subida, Subida,"
COGN3,Cogna,102051600,Education & Training Services,education-training-services,Education & Training Services,Consumer Defensive,consumer-defensive,Consumer Defensive,"24,187.00",NaN,...,1.94,32189600,0.00,0,1.94,COGN3,0.02,0.01,"2025-02-03, 2025-02-04, 2025-02-07, 2025-02-12...","Subida, Descida, Subida, Descida, Descida, Sub..."
MGLU3,Magazine Luiza,65687600,Specialty Retail,specialty-retail,Specialty Retail,Consumer Cyclical,consumer-cyclical,Consumer Cyclical,NaN,NaN,...,10.49,27433000,0.00,0,10.51,MGLU3,0.22,0.10,"2025-02-04, 2025-02-11, 2025-02-12, 2025-02-27...","Subida, Subida, Descida, Descida, Descida, Sub..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALPA3,Alpargatas,25500,Footwear & Accessories,footwear-accessories,Footwear & Accessories,Consumer Cyclical,consumer-cyclical,Consumer Cyclical,NaN,0.10,...,6.50,5700,0.00,0,6.63,ALPA3,0.02,0.03,"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-10...","Subida, Subida, Subida, Descida, Subida, Subid..."
INEP3,Inepar,25400,Specialty Industrial Machinery,specialty-industrial-machinery,Specialty Industrial Machinery,Industrials,industrials,Industrials,NaN,NaN,...,1.46,2100,0.00,0,1.45,INEP3,-0.00,0.00,"2025-02-07, 2025-02-12, 2025-02-24, 2025-02-26...","Subida, Subida, Subida, Subida, Subida, Subida..."
ENGI4,Energisa,24400,Utilities - Regulated Electric,utilities-regulated-electric,Utilities - Regulated Electric,Utilities,utilities,Utilities,"17,141.00",0.58,...,7.35,10200,0.00,0,7.33,ENGI4,0.05,0.02,"2025-02-12, 2025-02-26, 2025-03-14, 2025-03-20...","Descida, Subida, Descida, Descida, Subida,"


# Análise Exploratória

In [ ]:
analise_exploratoria(
    bd_dados_completos,
    imprime_todas_colunas = False,
    imprime_info_colunas = True,
    imprime_colunas_vazias = True,
    imprime_colunas_completas = True,
    imprime_colunas_parciais = True,
    remove_da_base = True,
    retorna_parciais = True,
    detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    )

<class 'pandas.core.frame.DataFrame'>
Index: 248 entries, B3SA3 to KRSA3
Data columns (total 20 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Nome da Empresa                                 248 non-null    object 
 1   Volume no último dia útil (lido em 16/03/2025)  248 non-null    int64  
 2   industry                                        248 non-null    object 
 3   industryKey                                     248 non-null    object 
 4   industryDisp                                    248 non-null    object 
 5   sector                                          248 non-null    object 
 6   sectorKey                                       248 non-null    object 
 7   sectorDisp                                      248 non-null    object 
 8   fullTimeEmployees                               106 non-null    float64
 9   dividendRate                              

None

<class 'pandas.core.frame.DataFrame'>
Index: 248 entries, B3SA3 to KRSA3
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   averageDailyVolume10Day       248 non-null    int64  
 1   marketCap                     248 non-null    int64  
 2   priceToSalesTrailing12Months  247 non-null    float64
 3   fiftyDayAverage               248 non-null    float64
 4   twoHundredDayAverage          248 non-null    float64
 5   trailingAnnualDividendRate    248 non-null    float64
 6   trailingAnnualDividendYield   248 non-null    float64
 7   profitMargins                 247 non-null    float64
 8   trailingEps                   247 non-null    float64
 9   forwardEps                    213 non-null    float64
 10  lastSplitFactor               153 non-null    object 
 11  lastSplitDate                 153 non-null    float64
 12  enterpriseToRevenue           244 non-null    float64
 13  ente

None

<class 'pandas.core.frame.DataFrame'>
Index: 248 entries, B3SA3 to KRSA3
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   numberOfAnalystOpinions  197 non-null    float64
 1   totalCash                247 non-null    float64
 2   totalCashPerShare        244 non-null    float64
 3   ebitda                   231 non-null    float64
 4   totalDebt                247 non-null    float64
 5   quickRatio               233 non-null    float64
 6   currentRatio             233 non-null    float64
 7   totalRevenue             247 non-null    float64
 8   debtToEquity             221 non-null    float64
 9   revenuePerShare          243 non-null    float64
 10  returnOnAssets           245 non-null    float64
 11  returnOnEquity           233 non-null    float64
 12  grossProfits             247 non-null    float64
 13  freeCashflow             230 non-null    float64
 14  operatingCashflow        

...

<class 'pandas.core.frame.DataFrame'>
Index: 248 entries, B3SA3 to KRSA3
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   2025-03-26; Ticker         248 non-null    object 
 1   Alfa HLC; últimos 13 dias  248 non-null    float64
 2   Alfa HLC; últimos 55 dias  248 non-null    float64
 3   Martelos                   248 non-null    object 
 4   Tipos de Martelos          248 non-null    object 
dtypes: float64(2), object(3)
memory usage: 19.7+ KB


None

Não há colunas NA
Colunas completas: ['Nome da Empresa', 'Volume no último dia útil (lido em 16/03/2025)', 'industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'marketCap', 'fiftyDayAverage', 'twoHundredDayAverage', 'trailingAnnualDividendRate', 'trailingAnnualDividendYield', '52WeekChange', 'SandP52WeekChange', 'recommendationKey', 'operatingMargins', 'fiftyDayAverageChange', 'fiftyDayAverageChangePercent', 'twoHundredDayAverageChange', 'twoHundredDayAverageChangePercent', '2025-01-31; Open', '2025-01-31; High', '2025-01-31; Low', '2025-01-31; Close', '2025-01-31; Volume', '2025-01-31; Dividends', '2025-01-31; Stock Splits', '2025-01-31; HLC', '2025-01-31; Ticker', '2025-02-03; Open', '2025-02-03; High', '2025-02-03; Low', '2025-02-03; Close', '2025-02-03; Volume', '2025-02-03; Dividends', '2025-02-03; Stock Splits', '2025-02-03; HLC', '2025-02-03; Ticker', '2

TypeError: list indices must be integers or slices, not str

: 